# Musanga V4 — NBAA: Annotation Consistency — exploratory audit

**Purpose:** explore *two distinct annotation tasks* before modelling: (1) five experts' `Musanga / non-Musanga / uncertain` classifications on the same candidate crowns, and (2) independently digitized Musanga-positive crown polygons in six verification circles by **four** annotators. Polygon absence is **not** automatically a non-Musanga vote.

**Scientific safeguards:** exact `verify_id` joins; no ground-truth claims or species inference; no models trained; no changes to V3_1/V4 reference or folds; spatial coordinates reprojected to UTM 35N before area/overlap calculations; independent polygon sets are *not* matched by `crown_id` (local per-annotator IDs). Comparisons of polygon overlap are restricted to circles both experts annotated. Shape uncertainty and annotation completeness are flagged, not silently resolved.

Read-only source: `Final/Data/Annotations/Verification/`. All outputs: `Final/Outputs/v4/NBAA_annotation_consistency/`.

**Run cells sequentially in Colab.** Large mosaics and LAS files are not accessed. This notebook is for initial scientific data reconnaissance, not annotator-specific classification training.

## 00 — V3_1-compatible Drive setup and file inventory

In [ ]:
%pip -q install geopandas shapely pyogrio scipy scikit-learn pandas matplotlib
from pathlib import Path
import json, re, math, warnings, itertools
from collections import Counter, defaultdict
import numpy as np, pandas as pd, geopandas as gpd, matplotlib.pyplot as plt
from shapely.geometry import shape
from shapely.ops import unary_union
from scipy.optimize import linear_sum_assignment
from sklearn.metrics import cohen_kappa_score
from IPython.display import display
from google.colab import drive
if not Path('/content/drive/MyDrive').exists(): drive.mount('/content/drive')
ROOT=next((p for p in [Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final'),
                         Path('/content/drive/MyDrive/Musanga/Final')] if p.is_dir()),None)
assert ROOT is not None,'Original Musanga Final directory unavailable'
DATA=ROOT/'Data'
SRC=DATA/'Annotations'/'Verification'
assert SRC.is_dir(),f'Expected input directory missing: {SRC}'
OUT=ROOT/'Outputs/v4/NBAA_annotation_consistency'
OUT.mkdir(parents=True,exist_ok=True)
files=sorted(p for p in SRC.iterdir() if p.is_file())
inventory=pd.DataFrame([dict(filename=p.name,extension=p.suffix.lower() or '(none)',
                             size_bytes=p.stat().st_size) for p in files])
inventory.to_csv(OUT/'NBAA_00_file_inventory.csv',index=False)
display(inventory)
print('Read-only source:',SRC)
print('Write-only NBAA output:',OUT)
print('Reference V3_1 / V4 datasets are NOT edited.')

## 01 — Classification CSV audit: exact crown IDs, labels and missingness

In [ ]:
CLASS_CSV=sorted(SRC.glob('musanga_classifications_*.csv'))
assert len(CLASS_CSV)==5, f'Expected 5 independent CSVs, found {[p.name for p in CLASS_CSV]}'
def who(p):
    n=p.name.casefold()
    if 'nadine' in n:return 'Nadine'
    if 'marjin' in n or 'marijn' in n:return 'Marijn'
    if 'joseph' in n:return 'Joseph'
    if 'thomas' in n:return 'Thomas'
    if 'kristof' in n:return 'Kristof'
    raise ValueError('Unknown annotator '+str(p))
NORM={'musanga':'M','non_musanga':'N','non-musanga':'N','non musanga':'N','uncertain':'U',
      'm':'M','n':'N','u':'U'}
raw_tables={};qa_rows=[];table_rows=[]
for p in CLASS_CSV:
    person=who(p);d=pd.read_csv(p,dtype={'verify_id':str})
    required={'verify_id','circle_id','classification'}
    assert required.issubset(d.columns),(p.name,required-set(d.columns))
    d['verify_id']=d.verify_id.astype(str).str.strip()
    d['label']=d.classification.astype(str).str.strip().str.casefold().map(NORM)
    qa_rows.append(dict(annotator=person,file=p.name,n_rows=len(d),
        unique_ids=d.verify_id.nunique(),duplicate_ids=int(d.verify_id.duplicated().sum()),
        invalid_labels=int(d.label.isna().sum()),circles=','.join(map(str,sorted(d.circle_id.dropna().unique()))),
        complete_flag_values=str(sorted(d.annotation_complete.dropna().astype(str).unique().tolist()))
             if 'annotation_complete' in d else '(absent)'))
    if d.verify_id.duplicated().any():raise ValueError(f'Duplicate verify_id in {p.name}')
    if d.label.isna().any():raise ValueError(f'Unexpected classification labels in {p.name}')
    raw_tables[person]=d
    table_rows.append(d[['verify_id','circle_id','label']].rename(columns={'circle_id':f'circle_{person}',
                                                                          'label':person}))
QA=pd.DataFrame(qa_rows).sort_values('annotator')
display(QA)
QA.to_csv(OUT/'NBAA_01_classification_file_audit.csv',index=False)
sets={who(p):set(raw_tables[who(p)].verify_id) for p in CLASS_CSV}
all_ids=set.union(*sets.values());common=set.intersection(*sets.values())
print('Union of IDs:',len(all_ids),'IDs present for every annotator:',len(common))
if len(common)!=len(all_ids):
    print('Missing IDs by annotator:',{name:sorted(all_ids-ids) for name,ids in sets.items()})
    print('Only complete matched IDs will enter five-expert agreement summaries.')
all_table=pd.DataFrame({'verify_id':sorted(all_ids)})
for t in table_rows:all_table=all_table.merge(t,on='verify_id',how='left',validate='one_to_one')
for person in raw_tables:
    derived=all_table.verify_id.str.extract(r'^C(\d+)_V\d+$')[0].astype('Int64')
    bad=all_table[f'circle_{person}'].notna()&(derived!=all_table[f'circle_{person}'].astype('Int64'))
    assert not bad.any(),f'circle_id vs verify_id disagreement for {person}'
all_table['circle_id']=all_table.verify_id.str.extract(r'^C(\d+)_V\d+$')[0].astype('Int64')
EXPERTS=['Nadine','Thomas','Joseph','Marijn','Kristof']
assert set(EXPERTS)==set(raw_tables)
matched=all_table.dropna(subset=EXPERTS).copy()
matched[['verify_id','circle_id',*EXPERTS]].to_csv(OUT/'NBAA_01_aligned_labels.csv',index=False)
display(matched.groupby('circle_id').size().rename('candidate_crowns').reset_index())
display(pd.DataFrame({person:matched[person].value_counts() for person in EXPERTS}).fillna(0).astype(int))
print('No label harmonization beyond the explicit M/N/U code mapping.')

## 02 — Classification consistency (descriptive, not annotator 'accuracy')

In [ ]:
stats=[]
for a,b in itertools.combinations(EXPERTS,2):
    q=matched[[a,b]].dropna();definite=q[(q[a]!='U')&(q[b]!='U')]
    stats.append(dict(annotator_a=a,annotator_b=b,n=len(q),
        agreement=float((q[a]==q[b]).mean()),
        cohen_kappa=float(cohen_kappa_score(q[a],q[b],labels=['M','N','U'])),
        n_definite=len(definite),
        definite_agreement=float((definite[a]==definite[b]).mean()) if len(definite) else np.nan,
        definite_kappa=float(cohen_kappa_score(definite[a],definite[b],labels=['M','N']))
           if len(definite) and definite[a].nunique()>1 and definite[b].nunique()>1 else np.nan))
PAIR=pd.DataFrame(stats)
PAIR.to_csv(OUT/'NBAA_02_pairwise_classifications.csv',index=False)
display(PAIR.round(3))
votes=matched[EXPERTS].apply(lambda r:r.value_counts(),axis=1).fillna(0).astype(int)
for k in ['M','N','U']:
    matched[f'votes_{k}']=votes[k].values if k in votes else 0
matched['largest_agreement']=matched[[f'votes_{x}' for x in 'MNU']].max(axis=1)
matched['majority_label']=matched.apply(lambda r: next((k for k in 'MNU' if r[f'votes_{k}']>=3),'none'),axis=1)
display(matched.groupby(['circle_id','largest_agreement']).size().unstack(fill_value=0))
display(matched.groupby('majority_label').size().rename('crowns'))
matched.to_csv(OUT/'NBAA_02_crown_vote_table.csv',index=False)
print('NOTE: agreement measures label similarity, NOT botanical species correctness.')

## 03 — Read the four independently digitized polygon sets (no suffix required)

In [ ]:
POLY_FILES=sorted(p for p in SRC.iterdir() if p.name.startswith('Musanga_Polygones_'))
print('Polygon export files:',[p.name for p in POLY_FILES])
assert len(POLY_FILES)==4, f'Expected 4 polygon exports; found {len(POLY_FILES)}'
POLY={}
poly_quality=[]
UTM='EPSG:32635'
for p in POLY_FILES:
    person=who(p)
    # These extensionless files were verified as GeoJSON FeatureCollections on Drive.
    # Do not infer shapefile structure from filename. Full text is small (<100 kB).
    raw=json.loads(p.read_text(encoding='utf-8-sig'))
    assert raw.get('type')=='FeatureCollection' and isinstance(raw.get('features'),list),p.name
    entries=[]
    for i,f in enumerate(raw['features']):
        props=f.get('properties') or {}
        geom=shape(f['geometry']) if f.get('geometry') else None
        entries.append({**props,'geometry':geom,'source_row':i})
    d=gpd.GeoDataFrame(entries,geometry='geometry',crs='EPSG:4326')
    assert 'circle_id' in d and 'crown_id' in d,p.name
    d['circle_id']=pd.to_numeric(d.circle_id,errors='raise').astype(int)
    invalid_before=int((~d.geometry.is_valid).sum())
    null_empty=int(d.geometry.isna().sum()+d.geometry.is_empty.sum())
    # Audit geometry only, no silent repair or clipping.
    d=d.to_crs(UTM)
    d['area_m2']=d.geometry.area
    d['centroid_x']=d.geometry.centroid.x;d['centroid_y']=d.geometry.centroid.y
    d['annotator']=person
    POLY[person]=d
    for cid,g in d.groupby('circle_id'):
        poly_quality.append(dict(annotator=person,circle_id=int(cid),n_polygons=len(g),
             sum_polygon_area_m2=float(g.area_m2.sum()),
             union_area_m2=float(unary_union(g.geometry.tolist()).area),
             median_polygon_area_m2=float(g.area_m2.median()),
             duplicate_crown_ids=int(g.crown_id.duplicated().sum()),
             invalid_before=invalid_before,null_or_empty=null_empty,
             annotation_status=str(g.annotation_status.value_counts().to_dict())
                 if 'annotation_status' in g else '(absent)'))
POLY_QA=pd.DataFrame(poly_quality).sort_values(['circle_id','annotator'])
POLY_QA.to_csv(OUT/'NBAA_03_polygon_coverage_by_circle.csv',index=False)
display(POLY_QA.round(2))
matrix=POLY_QA.pivot(index='circle_id',columns='annotator',values='n_polygons').fillna(0).astype(int)
display(matrix)
print('Total polygons:',{a:len(d) for a,d in POLY.items()})
print('Crucial: missing polygon != non-Musanga classification; completeness unknown unless documented.')
print('Also: crown_id strings in different files are NOT matched across annotators.')

## 04 — Pairwise *spatial* agreement where both annotators drew polygons

In [ ]:
# Area-level IoU on comparable circle-specific polygon unions.
# Detection overlap alone cannot distinguish two species without independent truth.
pair_rows=[]
for a,b in itertools.combinations(POLY.keys(),2):
    overlap_circles=sorted(set(POLY[a].circle_id)&set(POLY[b].circle_id))
    for cid in overlap_circles:
        aa=POLY[a].loc[POLY[a].circle_id==cid]
        bb=POLY[b].loc[POLY[b].circle_id==cid]
        ga=unary_union(aa.geometry.tolist());gb=unary_union(bb.geometry.tolist())
        inter=ga.intersection(gb).area;uni=ga.union(gb).area
        pair_rows.append(dict(circle_id=cid,annotator_a=a,annotator_b=b,
          n_a=len(aa),n_b=len(bb),area_a_m2=float(ga.area),area_b_m2=float(gb.area),
          intersection_m2=float(inter),union_m2=float(uni),
          area_iou=float(inter/uni) if uni>0 else np.nan,
          a_covered_by_b=float(inter/ga.area) if ga.area>0 else np.nan,
          b_covered_by_a=float(inter/gb.area) if gb.area>0 else np.nan))
SPATIAL=pd.DataFrame(pair_rows)
SPATIAL.to_csv(OUT/'NBAA_04_pairwise_polygon_union_IoU.csv',index=False)
display(SPATIAL.sort_values(['circle_id','annotator_a','annotator_b']).round(3))
display(SPATIAL.groupby(['annotator_a','annotator_b']).agg(
  circles=('circle_id','count'),mean_area_IoU=('area_iou','mean'),
  median_area_IoU=('area_iou','median')).round(3))
print('These are unsigned overlap measures; polygons can represent different crowns or delineation rules.')

## 05 — Crown-to-crown spatial matching (exploratory, threshold sensitivity)

In [ ]:
def match_iou(A,B,minimum):
    # IoU is computed in projected metres; optimal one-to-one Hungarian assignment.
    if not len(A) or not len(B):return dict(matches=0,iou_mean=np.nan,frac_a=0,frac_b=0)
    mat=np.zeros((len(A),len(B)),dtype=float)
    for i,ga in enumerate(A.geometry):
        for j,gb in enumerate(B.geometry):
            if not ga.intersects(gb):continue
            u=ga.union(gb).area
            mat[i,j]=ga.intersection(gb).area/u if u>0 else 0
    rr,cc=linear_sum_assignment(-mat)
    good=mat[rr,cc]>=minimum
    return dict(matches=int(good.sum()),
        iou_mean=float(mat[rr[good],cc[good]].mean()) if good.any() else np.nan,
        frac_a=float(good.sum()/len(A)),frac_b=float(good.sum()/len(B)))
match_rows=[]
for a,b in itertools.combinations(POLY.keys(),2):
    for cid in sorted(set(POLY[a].circle_id)&set(POLY[b].circle_id)):
        da=POLY[a].loc[POLY[a].circle_id==cid]
        db=POLY[b].loc[POLY[b].circle_id==cid]
        for threshold in (.10,.30,.50):
            match_rows.append(dict(circle_id=cid,a=a,b=b,iou_threshold=threshold,
                n_a=len(da),n_b=len(db),**match_iou(da,db,threshold)))
MATCH=pd.DataFrame(match_rows)
MATCH.to_csv(OUT/'NBAA_05_one_to_one_crown_matches.csv',index=False)
display(MATCH.groupby(['a','b','iou_threshold']).agg(
    circles=('circle_id','count'),matched_crowns=('matches','sum'),
    avg_fraction_a=('frac_a','mean'),avg_fraction_b=('frac_b','mean')).round(3))
print('Unmatched crown need not be a false positive: reference completeness is unverified.')

## 06 — Compact site map / visual QA and exploration hand-off

In [ ]:
# Small vector-only visualization: no large TIFF reads.
from matplotlib.patches import Patch
colors={'Nadine':'#1e7962','Thomas':'#cf7c37','Joseph':'#6178b8','Marijn':'#ad589c'}
circles=sorted(set.union(*(set(d.circle_id) for d in POLY.values())))
fig,axes=plt.subplots(2,3,figsize=(15,10))
for ax,cid in zip(axes.flat,circles):
    for person in ['Nadine','Thomas','Joseph','Marijn']:
        d=POLY.get(person)
        if d is None:continue
        subset=d[d.circle_id==cid]
        if len(subset):subset.boundary.plot(ax=ax,color=colors[person],linewidth=1,alpha=.8)
    ax.set_title(f'Circle {cid} — independently digitized crowns')
    ax.set_aspect('equal');ax.tick_params(labelsize=7)
for ax in axes.flat[len(circles):]:ax.axis('off')
fig.legend(handles=[Patch(facecolor='none',edgecolor=colors[n],label=n) for n in colors],
           loc='lower center',ncol=4)
plt.tight_layout(rect=[0,.04,1,1])
fig.savefig(OUT/'NBAA_06_polygon_overlay_by_circle.png',dpi=170)
plt.show();plt.close(fig)
contract=dict(schema='NBAA_v4_exploratory_1',
  source_directory=str(SRC),output_directory=str(OUT),
  classification_files={p:raw_tables[p].shape[0] for p in EXPERTS},
  common_classification_crowns=int(len(matched)),
  digitized_polygon_counts={p:int(len(POLY[p])) for p in POLY},
  classification_policy='Uncertain is a separate observed label; missing is not negative',
  polygon_policy='Independent positive polygons; absent polygons not negative; no automatic truth',
  alignment='CSV verify_id joins; polygon overlays by projected geometry and circle',
  train_test_policy='No training or new splits in this audit',
  further_review=['Digitizing completeness for each circle and annotator',
                  'Compare classification vs polygon tasks only where identity/coverage established',
                  'Independent RGB/DSM embeddings and spatially held-out annotator predictors',
                  'Repeat blinded annotations to assess genuine intra-annotator repeatability'])
(OUT/'NBAA_manifest.json').write_text(json.dumps(contract,indent=2))
print('NBAA exploration complete. Outputs:')
for p in sorted(OUT.glob('NBAA_*')):print(' ',p.name)
print('NEXT DECISION: review counts, completeness, overlap and image QA before modelling.')